In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data

In [4]:
# Load datasets — adjust path if your files are in "titanic/" instead of "downloads/"
train_dataset = pd.read_csv("downloads/train.csv")
predict_dataset = pd.read_csv("downloads/test.csv")

print(f"Train shape: {train_dataset.shape}")

Train shape: (891, 12)


## 2. Explore the Data

In [5]:
train_dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
train_dataset.info()
predict_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64

In [7]:
# Check missing values in train
# print("=== Train missing values ===")
print(train_dataset.isnull().sum())
print("predict_test")
print(predict_dataset.isnull().sum())
print(f"\nTotal rows: {len(train_dataset)}")

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
predict_test
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Total rows: 891


## 3. Extract Target & Prepare Features

In [8]:
# --- Extract target ---
# Train target
survived_train = train_dataset["Survived"].copy()

print(f"Train target shape: {survived_train.shape}")
print(f"\nTrain target distribution:\n{survived_train.value_counts()}")

Train target shape: (891,)

Train target distribution:
Survived
0    549
1    342
Name: count, dtype: int64


## 4. Handle Missing Values

In [9]:
# Work on copies
train_df = train_dataset.copy()
predict_df = predict_dataset.copy()

# Fill Age with median (computed from train only to avoid data leakage)
age_median = train_df["Age"].median()
train_df["Age"] = train_df["Age"].fillna(age_median)
predict_df["Age"] = predict_df["Age"].fillna(age_median)

# Fill Embarked with mode (from train)
embarked_mode = train_df["Embarked"].mode()[0]
train_df["Embarked"] = train_df["Embarked"].fillna(embarked_mode)

# Fill Fare with median (from train) — test has 1 missing
fare_median = train_df["Fare"].median()
train_df["Fare"] = train_df["Fare"].fillna(fare_median)
predict_df["Fare"] = predict_df["Fare"].fillna(fare_median)

print("After filling — remaining nulls in train:")
print(train_df[["Age", "Embarked", "Fare"]].isnull().sum())
print("\nAfter filling — remaining nulls in test:")

After filling — remaining nulls in train:
Age         0
Embarked    0
Fare        0
dtype: int64

After filling — remaining nulls in test:


## 5. Feature Engineering

We'll create a feature called `under_age` (Age < 18) and then encode categorical columns using `OrdinalEncoder`.

In [10]:
# Create under_age feature
train_df["under_age"] = (train_df["Age"] < 18).astype(int)
predict_df["under_age"] = (predict_df["Age"] < 18).astype(int)

# Select the columns we'll use
# Categorical columns to encode
cat_cols = ["Sex", "Embarked"]

# Numeric columns (already numeric)
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare", "under_age"]

# All feature columns
feature_cols = num_cols + cat_cols

print("Feature columns:", feature_cols)
print(f"\nTrain categorical sample:")
print(train_df[cat_cols].head())

Feature columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'under_age', 'Sex', 'Embarked']

Train categorical sample:
      Sex Embarked
0    male        S
1  female        C
2  female        S
3  female        S
4    male        S


In [11]:
# --- Ordinal Encoding for categorical features ---
# Fit on train, transform both train and test
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Fit on train categorical columns
ordinal_encoder.fit(train_df[cat_cols])

# Transform
train_df[cat_cols] = ordinal_encoder.transform(train_df[cat_cols])
predict_df[cat_cols] = ordinal_encoder.transform(predict_df[cat_cols])

# Show the encoding categories
for col, cats in zip(cat_cols, ordinal_encoder.categories_):
    print(f"{col}: {list(cats)} -> {list(range(len(cats)))}")

print(f"\nTrain encoded sample:")
print(train_df[cat_cols].head())
print(predict_df[cat_cols].head())

Sex: ['female', 'male'] -> [0, 1]
Embarked: ['C', 'Q', 'S'] -> [0, 1, 2]

Train encoded sample:
   Sex  Embarked
0  1.0       2.0
1  0.0       0.0
2  0.0       2.0
3  0.0       2.0
4  1.0       2.0
   Sex  Embarked
0  1.0       1.0
1  0.0       2.0
2  1.0       1.0
3  1.0       2.0
4  0.0       2.0


In [12]:
# Build final feature matrices
X_train = train_df[feature_cols][:850].values
y_train = survived_train[:850].values

x_test = train_df[feature_cols][850:].values
y_test = survived_train[850:].values

predict_test = predict_df[feature_cols].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {x_test.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"pre_df shape:  {predict_test.shape}")

X_train shape: (850, 8)
y_train shape: (850,)
X_test shape:  (41, 8)
y_test shape:  (41,)
pre_df shape:  (418, 8)


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
# Train with cross-validation on the training set
rf_clf = RandomForestClassifier()
gscv = GridSearchCV(rf_clf,[{"n_estimators":[1,5,100,150,50],"max_depth":[6,9,10,20,16,50]}],scoring="accuracy",cv=8)
gscv.fit(X_train,y_train)
y_pred = gscv.predict(x_test)
test_accuracy = accuracy_score(y_test, y_pred)
accuracy_score(y_test,gscv.predict(x_test))

0.8536585365853658

## 7. Evaluate on Test Set

In [14]:
# Fit on full training set, then predict on test
rf_clf.fit(X_train, y_train)
y_pred = gscv.predict(predict_test)
test_accuracy = accuracy_score(y_train[:418], y_pred[:418])
print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.5167


In [15]:
dataset = pd.DataFrame([],columns=["PassengerId","Survived"])
dataset["PassengerId"] = predict_dataset["PassengerId"]
dataset["Survived"] = y_pred
dataset

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [16]:
dataset.to_csv("output.csv",index=False)

In [17]:
predict_dataset.PassengerId,y_pred

(0       892
 1       893
 2       894
 3       895
 4       896
        ... 
 413    1305
 414    1306
 415    1307
 416    1308
 417    1309
 Name: PassengerId, Length: 418, dtype: int64,
 array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1,
        1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
        1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
        1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
        1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
        0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
        0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
        1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1,